# MMCultureQA – Task 2 (Textual VQA): fine-tuning QLoRA de Qwen2.5-Omni-3B · versión Kaggle

Versión para **Kaggle Notebooks** (GPU T4 gratis, unas 30 h por semana, sesiones de hasta 12 h). Hace lo mismo que el notebook de Colab, pero sin Google Drive: los datos vienen de Hugging Face y los resultados se guardan en `/kaggle/working`.

### Antes de empezar (panel derecho → *Settings*)
1. **Accelerator: GPU T4 x2.** Se usa solo una de las dos GPU, igual que en Colab.
2. **Internet: On.** Hace falta para Hugging Face y GitHub; la cuenta de Kaggle tiene que estar verificada con el teléfono.
3. **Adaptador ya entrenado:** *Add Input* → un dataset con la carpeta `models/` del proyecto (contiene `models/qlora_r16_n2000/adapter_final/`), o el *output* de una ejecución anterior de este notebook. La celda 2 encuentra `adapter_final` solo en `/kaggle/input`.

Por defecto (`TRAIN = False`) el notebook **no entrena**: carga ese adaptador y solo evalúa. Para entrenar uno nuevo: `TRAIN = True`. Si además hay un checkpoint en el input (`state.pt`, `adapter_model.safetensors`, `adapter_config.json`), el entrenamiento continúa desde ese paso.

### Cómo ejecutarlo
- **Recomendado: *Save Version → Save & Run All (Commit)*.** Se ejecuta en segundo plano hasta 12 h, aunque cierres el navegador. Al terminar, todo lo de `/kaggle/working` queda guardado como *output* de esa versión (adaptador, checkpoints, predicciones, scores).
- *Run All* en modo interactivo también funciona, pero la sesión se corta si la dejas inactiva y entonces `/kaggle/working` se pierde.

Pasos:

1. **Instalación** (sin reiniciar el kernel).
2. **Configuración**: busca el adaptador ya entrenado (o, con `TRAIN = True`, un checkpoint) en `/kaggle/input`.
3. **Datos**: `train` (para entrenar), `dev` (para evaluar) y las imágenes, directamente desde Hugging Face (las de `train` solo con `TRAIN = True`).
4. **Baseline v1**: limpia las respuestas antiguas de la baseline y las vuelve a evaluar.
5. **Modelo y LoRA**.
6. **Entrenamiento** en `train` (solo con `TRAIN = True`), continuando desde el checkpoint si lo hay.
7. **Evaluación** en `dev` con el adaptador ya entrenado (o el recién entrenado), con el scorer oficial (BERTScore-F1, mmBERT capa 17).

Valor de referencia: la baseline v0 obtiene **BERTScore-F1 = 0.7392** en `dev` (`results/scores_v0.json`).

**Importante:** solo se entrena con `train` y solo se evalúa con `dev`. El modelo nunca ve `dev` durante el entrenamiento.

**Para continuar un entrenamiento:** `N_TRAIN`, `N_VAL`, `SEED`, `GRAD_ACCUM` y `LORA_R` tienen que ser exactamente iguales que en la ejecución que creó el checkpoint (por defecto: 2000 / 200 / 42 / 8 / 16, como en Colab).

## 1. Instalación

In [ ]:
import os, socket, subprocess, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle da 2× T4; usamos una, como en Colab (antes de importar torch)

# Sin internet no se puede instalar nada ni descargar datos: parar aquí con un mensaje claro.
for host in ("pypi.org", "huggingface.co"):
    try:
        socket.create_connection((host, 443), timeout=10).close()
    except OSError:
        raise RuntimeError(f"Sin internet ({host} no responde): panel derecho → Session options → Internet → On. "
                           "Si el interruptor no aparece: perfil → Settings → Phone Verification.")

# En Kaggle no hace falta reiniciar: transformers todavía no está importado en este kernel.
# torch y torchvision ya vienen instalados en Kaggle y no se tocan.
def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])   # falla con error si pip falla

pip("-U", "transformers>=4.52.0,<5", "peft>=0.13", "accelerate", "bitsandbytes", "qwen-omni-utils",
    "soundfile", "librosa", "datasets", "huggingface_hub[hf_transfer]", "pyarrow")
pip("bert_score==0.3.12", "sacrebleu==2.5.1", "rouge-score", "sentencepiece")
print("Paquetes listos")

## 2. Configuración y checkpoint

In [ ]:
import os, glob, shutil, socket
import torch
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
except OSError:
    raise RuntimeError("Sin internet: Settings → Internet → On (la cuenta de Kaggle tiene que estar verificada)")

TASK, TRACK, REGION = "qa", "en", "mena"
REPO   = "QCRI/MMCQA-SemEval27"
CONFIG = f"{TASK}_{REGION}_{TRACK}"

MODEL      = "Qwen/Qwen2.5-Omni-3B"
MAX_PIXELS = 512 * 28 * 28     # igual que la baseline y que en Colab
ROOT       = "/tmp/mmcqa"      # imágenes descomprimidas (fuera de /kaggle/working, para no llenar el output)
IMG_INPUT  = "/kaggle/input/mmcqa-images"   # opcional: un dataset de Kaggle con images/… o los ZIP; si no existe, se descarga de Hugging Face

SMOKE_TEST = False   # True = prueba corta (≈ 10–15 min) para comprobar que todo funciona

# False (por defecto) = NO entrenar: cargar el adaptador LoRA ya entrenado desde /kaggle/input y evaluarlo.
# True = entrenar un adaptador nuevo (o continuar desde un checkpoint en /kaggle/input).
TRAIN = False
ADAPTER_NAME = "qlora_r16_n2000"   # igual que models/qlora_r16_n2000/adapter_final en el proyecto

# --- Entrenamiento (para continuar un checkpoint: igual que en la ejecución original) ---
N_TRAIN     = 2000
N_VAL       = 200
EPOCHS      = 1
LR          = 1e-4
GRAD_ACCUM  = 8
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
MAX_LEN     = 1280
SAVE_EVERY  = 50
EVAL_EVERY  = 50
SEED        = 42

# --- Evaluación ---
EVAL_LIMIT  = None      # None = las 1000 preguntas de dev (≈ 10 s por pregunta en una T4, ≈ 3 h); p. ej. 300
EVAL_PROMPT_ONLY = True # True = evaluar también v2 (prompt nuevo sin LoRA, para comparar; tarda lo mismo que v3).
                        # False = saltar v2 (con EVAL_LIMIT = 300 se usan los valores medidos antes)
MAX_NEW     = 64

if SMOKE_TEST:
    N_TRAIN, N_VAL, EVAL_LIMIT, SAVE_EVERY, EVAL_EVERY = 48, 8, 20, 3, 3

RUN_NAME = f"qlora_r{LORA_R}_n{N_TRAIN}" + ("_smoketest" if SMOKE_TEST else "")
OUT_DIR  = f"/kaggle/working/{RUN_NAME}"
CKPT     = f"{OUT_DIR}/checkpoint"     # adaptador LoRA + estado del optimizador
os.makedirs(OUT_DIR, exist_ok=True)
print(CONFIG, "->", OUT_DIR)

# --- Importar checkpoint / adaptador desde /kaggle/input ---
CKPT_FILES = ["state.pt", "adapter_model.safetensors", "adapter_config.json"]

def ckpt_step(d):
    path = f"{d}/state.pt"
    try:
        return torch.load(path, map_location="cpu", weights_only=False)["step"]
    except Exception as exc:          # no ocultar el error: un checkpoint ilegible no debe pasar desapercibido
        size = os.path.getsize(path) / 1e6 if os.path.isfile(path) else 0
        raise RuntimeError(f"No se puede leer {path} ({size:.1f} MB; el original pesa ≈ 229 MB): "
                           f"{type(exc).__name__}: {exc}") from exc

def pick(paths):
    # primero los que llevan el nombre de esta ejecución; los de la prueba corta solo si SMOKE_TEST
    same = [p for p in paths if RUN_NAME in p]
    if same:
        return same
    return [p for p in paths if ("smoketest" in p) == SMOKE_TEST]

# El checkpoint solo interesa para entrenar
found = pick(sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/state.pt", recursive=True)})) if TRAIN else []
if found:
    steps = {d: ckpt_step(d) for d in found}
    best = max(steps, key=steps.get)
    have = ckpt_step(CKPT) if os.path.isfile(f"{CKPT}/state.pt") else -1
    if steps[best] > have:
        missing = [f for f in CKPT_FILES if not os.path.isfile(f"{best}/{f}")]
        assert not missing, f"Al checkpoint en {best} le faltan: {missing}"
        os.makedirs(CKPT, exist_ok=True)
        for f in CKPT_FILES:
            shutil.copy(f"{best}/{f}", f"{CKPT}/{f}")
        print(f"Checkpoint importado: {best}  (paso {steps[best]})")
    else:
        print(f"Checkpoint en /kaggle/working ya está al día (paso {have})")
elif TRAIN:
    print("No hay checkpoint en /kaggle/input: el entrenamiento empieza desde cero")

# --- Adaptador ya entrenado (TRAIN = False) ---
cands = sorted(os.path.dirname(p) for p in
               glob.glob("/kaggle/input/**/adapter_final/adapter_model.safetensors", recursive=True))
cands = [p for p in cands if "smoketest" not in p]
cands = [p for p in cands if ADAPTER_NAME in p] or cands
ADAPTER_SRC = cands[0] if cands else None
if not TRAIN:
    assert ADAPTER_SRC, ("No hay ningún adapter_final en /kaggle/input. Add Input → dataset con la carpeta models/ "
                         "del proyecto (o el output de una ejecución anterior), o pon TRAIN = True.")
    print("TRAIN = False: se usará el adaptador ya entrenado:", ADAPTER_SRC)

## 3. Datos

`train` tiene 10.000 preguntas con respuesta y `dev` tiene 1.000. Las filas se cargan desde Hugging Face y las imágenes también (los ZIP de `archives/`). **No hace falta subir nada a Kaggle.**

Opcional: si añades como input un dataset de Kaggle llamado `mmcqa-images` (con `images/…` o con los ZIP), se usa ese en lugar de la descarga. De los ZIP solo se extraen las imágenes que de verdad hacen falta.

In [ ]:
import random, zipfile, shutil
from datasets import load_dataset
from huggingface_hub import hf_hub_download

train_all = load_dataset(REPO, CONFIG, split="train").to_list()
dev_rows  = load_dataset(REPO, CONFIG, split="dev").to_list()

rng = random.Random(SEED)
rng.shuffle(train_all)
val_rows   = train_all[:N_VAL]
train_rows = train_all[N_VAL:N_VAL + N_TRAIN]
print(f"train: {len(train_rows)} | val (de train): {len(val_rows)} | dev: {len(dev_rows)}")

def unpack(split, rows):
    wanted = {r["image"] for r in rows}
    todo = {m for m in wanted if not os.path.isfile(os.path.join(ROOT, m))}
    # 1) copiar imágenes sueltas desde el dataset de Kaggle (si existe)
    n_local = 0
    for m in sorted(todo):
        src = os.path.join(IMG_INPUT, m)
        if os.path.isfile(src):
            os.makedirs(os.path.dirname(os.path.join(ROOT, m)), exist_ok=True)
            shutil.copy(src, os.path.join(ROOT, m))
            n_local += 1
    todo = {m for m in todo if not os.path.isfile(os.path.join(ROOT, m))}
    n_zip, source = 0, "-"
    if todo:
        # 2) ZIP en el dataset de Kaggle, 3) si no, ZIP de Hugging Face
        name = f"images_{REGION}_{split}.zip"
        local = [p for p in (f"{IMG_INPUT}/{name}", f"{IMG_INPUT}/archives/{name}") if os.path.isfile(p)]
        path = local[0] if local else hf_hub_download(REPO, f"archives/{name}", repo_type="dataset")
        source = "Kaggle input" if local else "Hugging Face"
        with zipfile.ZipFile(path) as zf:
            for m in zf.namelist():
                if m in todo:
                    zf.extract(m, ROOT)
                    n_zip += 1
    print(f"{split}: {len(wanted)} imágenes necesarias | {n_local} copiadas del dataset de Kaggle | "
          f"{n_zip} del ZIP ({source}) | el resto ya estaba en {ROOT}")

if TRAIN:                            # las imágenes de train (ZIP de 1,7 GB) solo hacen falta para entrenar
    unpack("train", train_rows + val_rows)
unpack("dev", dev_rows)
needed = (train_rows + val_rows if TRAIN else []) + dev_rows
missing = [r for r in needed if not os.path.isfile(os.path.join(ROOT, r["image"]))]
assert not missing, f"Faltan {len(missing)} imágenes, p. ej. {missing[0]['image']}"

# Comprobar solapamiento: ¿aparecen las mismas imágenes/preguntas en train y en dev?
all_train_ids  = {r["id"] for r in train_all}
all_train_imgs = {r["image"] for r in train_all}
all_train_q    = {(r["question"] or "").strip().lower() for r in train_all}
print("IDs de dev también en train:          ", sum(r["id"] in all_train_ids for r in dev_rows))
print("Imágenes de dev también en train:     ", sum(r["image"] in all_train_imgs for r in dev_rows))
print("Preguntas de dev idénticas en train:  ", sum((r["question"] or "").strip().lower() in all_train_q for r in dev_rows))

import statistics
print("Longitud de las respuestas de referencia (palabras): mediana",
      statistics.median(len(r["answer"].split()) for r in train_all))
train_rows[0]

## 4. Limpiar respuestas y baseline v1

La baseline v0 a menudo sigue hablando: hace preguntas de vuelta („So, what do you think…?“) e incluso se inventa nuevos turnos de diálogo („Human: …“). Eso baja la **precision** de BERTScore (0,69 frente a 0,80 de recall).

Por eso `clean_answer` elimina:
- todo lo que viene a partir de `Human:` / `User:`
- las preguntas de vuelta (frases que terminan en `?`)
- una última frase cortada a la mitad por `max_new_tokens`
- muletillas al principio („Well, you know, …“)

Aquí se limpian las **respuestas antiguas de v0** y se vuelven a evaluar. No necesita GPU y muestra cuánto aporta solo la limpieza. Ese número es la **baseline v1**, la referencia justa para comparar el fine-tuning.

In [ ]:
import re, json

STOP_MARKERS = ["\nHuman:", "Human:", "\nUser:", "\nAssistant:", "<|im_end|>"]
FILLER = re.compile(r"^(?:(?:well|oh|hmm+|um+|uh+|you know|so|okay|ok|ah|yeah|alright)[,!.]?\s+)+", re.I)

def clean_answer(text: str) -> str:
    for m in STOP_MARKERS:                       # 1) quitar todo desde un turno de diálogo inventado
        i = text.find(m)
        if i != -1:
            text = text[:i]
    text = " ".join(text.split())
    sents = re.split(r"(?<=[.!?])\s+", text)
    if len(sents) > 1 and not re.search(r"[.!?]$", sents[-1]):
        sents = sents[:-1]                       # 2) última frase cortada
    kept = [s for s in sents if not s.endswith("?")] or sents[:1]   # 3) preguntas de vuelta
    out = FILLER.sub("", " ".join(kept).strip())  # 4) muletillas al principio
    return out[:1].upper() + out[1:] if out else text.strip()

print(clean_answer("Well, you know, it's the Burj Khalifa. So, what do you think?\nHuman: Next question"))

In [ ]:
# scorer oficial + nuestro repo (para las predicciones de v0)
!git clone -q https://github.com/mmcultureqa-semeval27/mmcultureqa.git /tmp/mmcultureqa 2>/dev/null || git -C /tmp/mmcultureqa pull -q
!git clone -q https://github.com/Nachotst/Task-6-Multilingual-and-Multimodal-Culturally-Grounded-Spoken-and-Visual-Question-Answering.git /tmp/project 2>/dev/null || git -C /tmp/project pull -q

GOLD = f"/tmp/gold_dev_{TRACK}.jsonl"
with open(GOLD, "w", encoding="utf-8") as fh:
    for r in dev_rows:
        fh.write(json.dumps({"id": r["id"], "answer": r["answer"]}, ensure_ascii=False) + "\n")

def write_preds(path, answers):
    with open(path, "w", encoding="utf-8") as fh:
        for i, a in answers.items():
            fh.write(json.dumps({"id": i, "answer": a, "lang": TRACK}, ensure_ascii=False) + "\n")

def score(pred_path, name):
    out = f"{OUT_DIR}/scores_{name}.json"
    !python /tmp/mmcultureqa/scorer/score.py --gold {GOLD} --pred {pred_path} --lang {TRACK} --out {out}
    with open(out) as fh:
        res = json.load(fh)["submission_result"]
    print(f"==> {name}: BERTScore-F1 = {res['BERTScore-F1']:.4f}")
    return res

v0 = [json.loads(l) for l in open("/tmp/project/results/prediction_qa_en_full.jsonl", encoding="utf-8")]
write_preds(f"{OUT_DIR}/prediction_v0.jsonl", {p["id"]: p["answer"] for p in v0})
write_preds(f"{OUT_DIR}/prediction_v1_clean.jsonl", {p["id"]: clean_answer(p["answer"]) for p in v0})
scores = {}
scores["v0 (baseline)"]          = score(f"{OUT_DIR}/prediction_v0.jsonl", "v0")
scores["v1 (v0 + limpieza)"]     = score(f"{OUT_DIR}/prediction_v1_clean.jsonl", "v1_clean")

## 5. Modelo y LoRA

In [ ]:
import torch
from transformers import BitsAndBytesConfig, Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration
from peft import LoraConfig, get_peft_model, PeftModel

assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU T4"
print("GPU:", torch.cuda.get_device_name(0))
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16   # T4: float16

quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
processor = Qwen2_5OmniProcessor.from_pretrained(MODEL, max_pixels=MAX_PIXELS)
base = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    MODEL, torch_dtype=DTYPE, device_map="auto", quantization_config=quant, attn_implementation="sdpa")

for p in base.parameters():          # congelar todo; solo se entrenan los pesos LoRA
    p.requires_grad_(False)
base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
base.config.use_cache = False

# LoRA solo en el modelo de lenguaje (model.layers.*), no en el encoder de visión ni en el de audio
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
                  target_modules=r"^model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)$")
model = get_peft_model(base, lora)
for n, p in model.named_parameters():  # pesos LoRA en float32: más estable con autocast float16
    if p.requires_grad:
        p.data = p.data.float()
n_lora = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A"))
print("Módulos LoRA:", n_lora)        # esperado: 36 capas × 7 = 252
assert n_lora > 0, "LoRA no encontró módulos: revisa target_modules (print(model) muestra los nombres)"
model.print_trainable_parameters()

## 6. Prompt y ejemplos de entrenamiento

El prompt es **el mismo en el entrenamiento y en la evaluación**, y eso es importante. Pide directamente una respuesta breve y factual, sin preguntas de vuelta.

Al entrenar, el modelo solo aprende la **respuesta** (`labels`). El system prompt, la imagen y la pregunta van enmascarados con `-100`.

In [ ]:
from qwen_omni_utils import process_mm_info

SYSTEM = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of "
          "perceiving auditory and visual inputs, as well as generating text and speech.")
LANGUAGE = {"en": "English", "msa": "Modern Standard Arabic", "arz": "Egyptian Arabic", "ajp": "Levantine Arabic"}
INSTRUCTION = ("Answer the question about the image in {language} with one or two complete, factual sentences. "
               "Name the specific place, object, food or tradition when you can. "
               "Do not ask questions and do not add comments.")

def build_conversation(row, answer=None):
    content = [{"type": "image", "image": os.path.join(ROOT, row["image"])},
               {"type": "text", "text": row["question"]},
               {"type": "text", "text": INSTRUCTION.format(language=LANGUAGE.get(TRACK, "the language of the question"))}]
    conv = [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": content}]
    if answer is not None:
        conv.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return conv

def encode_train(row):
    conv = build_conversation(row, row["answer"])
    prompt_text = processor.apply_chat_template(conv[:-1], add_generation_prompt=True, tokenize=False)
    full_text   = processor.apply_chat_template(conv, add_generation_prompt=False, tokenize=False)
    assert full_text.startswith(prompt_text), "Chat template: el prompt no es prefijo del texto completo"
    _, images, _ = process_mm_info(conv, use_audio_in_video=False)
    inputs = processor(text=full_text, images=images, return_tensors="pt", padding=False)
    answer_ids = processor.tokenizer(full_text[len(prompt_text):], add_special_tokens=False)["input_ids"]
    labels = inputs["input_ids"].clone()
    labels[:, :-len(answer_ids)] = -100          # solo la respuesta (+ <|im_end|>) cuenta para la pérdida
    inputs["labels"] = labels
    return inputs

ex = encode_train(train_rows[0])
print("Tokens:", ex["input_ids"].shape[1], "| de ellos respuesta:", int((ex["labels"] != -100).sum()))
print("Parte aprendida:", repr(processor.tokenizer.decode(ex["input_ids"][0][ex["labels"][0] != -100])))

## 7. Entrenamiento

El entrenamiento usa un bucle propio: batch 1, gradient accumulation, autocast float16 y GradScaler. Cada `SAVE_EVERY` pasos se guardan en `/kaggle/working` el adaptador y el estado del optimizador. **Si la ejecución se corta:** añade el *output* de esa versión como input (*Add Input → Notebook outputs*) y vuelve a ejecutar; continúa desde el último checkpoint. Para no perder la correspondencia con el checkpoint, no cambies `N_TRAIN`, `N_VAL`, `SEED` ni `GRAD_ACCUM` hasta que termine.

Con `TRAIN = False` (celda 2, por defecto) esta celda no entrena, y la sección 8 evalúa el adaptador ya entrenado (`ADAPTER_SRC`).

In [ ]:
import time, math
import torch.nn.functional as F
from peft import set_peft_model_state_dict
from safetensors.torch import load_file

def to_device(inputs):
    out = {}
    for k, v in inputs.items():
        v = v.to(model.device)
        out[k] = v.to(DTYPE) if v.is_floating_point() else v
    return out

def loss_on(inputs):
    inputs = to_device(inputs)
    labels = inputs.pop("labels")
    with torch.autocast("cuda", dtype=DTYPE):
        logits = model(**inputs, use_cache=False).logits
    return F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.size(-1)),
                           labels[:, 1:].reshape(-1), ignore_index=-100)

@torch.no_grad()
def val_loss():
    model.eval()
    losses = []
    for r in val_rows:
        ex = encode_train(r)
        if ex["input_ids"].shape[1] <= MAX_LEN:
            losses.append(loss_on(ex).item())
    model.train()
    return sum(losses) / len(losses)

if TRAIN:
    params = [p for p in model.parameters() if p.requires_grad]
    opt    = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
    total_steps  = math.ceil(len(train_rows) * EPOCHS / GRAD_ACCUM)
    warmup_steps = max(1, int(0.05 * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warmup_steps)
                                              * max(0.0, (total_steps - s) / max(1, total_steps - warmup_steps)))

    step, history = 0, []
    if os.path.exists(f"{CKPT}/state.pt"):        # continuar después de una interrupción
        set_peft_model_state_dict(model, load_file(f"{CKPT}/adapter_model.safetensors"))
        st = torch.load(f"{CKPT}/state.pt", weights_only=False)
        opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        step, history = st["step"], st["history"]
        print(f"Checkpoint encontrado: se continúa desde el paso {step}/{total_steps}")

    def save_ckpt():
        model.save_pretrained(CKPT)
        torch.save({"opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                    "step": step, "history": history}, f"{CKPT}/state.pt")

    order = [r for _ in range(EPOCHS) for r in train_rows]   # orden fijo, para que la reanudación cuadre
    model.train()
    if step == 0 and not history:
        history.append({"step": 0, "val_loss": val_loss()})
        print(f"Pérdida de validación antes de entrenar: {history[-1]['val_loss']:.4f}")

    t0, running, skipped = time.time(), [], 0
    for i in range(step * GRAD_ACCUM, len(order)):
        ex = encode_train(order[i])
        if ex["input_ids"].shape[1] > MAX_LEN:
            skipped += 1
        else:
            loss = loss_on(ex) / GRAD_ACCUM
            scaler.scale(loss).backward()
            running.append(loss.item() * GRAD_ACCUM)
        if (i + 1) % GRAD_ACCUM == 0 or i + 1 == len(order):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            step += 1
            if step % 10 == 0:
                print(f"Paso {step}/{total_steps} | pérdida train {sum(running[-80:]) / max(1, len(running[-80:])):.4f}"
                      f" | LR {sched.get_last_lr()[0]:.2e} | {(time.time() - t0) / 60:.1f} min", flush=True)
            if step % EVAL_EVERY == 0 or step == total_steps:
                history.append({"step": step, "val_loss": val_loss(),
                                "train_loss": sum(running[-80:]) / max(1, len(running[-80:]))})
                print(f"   Pérdida de validación: {history[-1]['val_loss']:.4f}")
            if step % SAVE_EVERY == 0 or step == total_steps:
                save_ckpt()
                print(f"   Checkpoint guardado ({CKPT})")
        if i % 50 == 0:
            torch.cuda.empty_cache()

    model.save_pretrained(f"{OUT_DIR}/adapter_final")
    json.dump(history, open(f"{OUT_DIR}/history.json", "w"), indent=1)
    print(f"Terminado. Saltados (demasiado largos): {skipped}. Adaptador: {OUT_DIR}/adapter_final")
else:
    print("TRAIN = False: se salta el entrenamiento (se evaluará el adaptador ya entrenado)")

## 8. Evaluación en `dev`

Dos ejecuciones con el mismo prompt nuevo:
- **sin adaptador** (`prompt_only`): muestra lo que aporta solo el prompt mejorado
- **con adaptador** (`finetuned`): muestra lo que aporta además el fine-tuning

Las respuestas se escriben en `/kaggle/working` a medida que se generan. Si el entorno se desconecta, la celda continúa desde donde se quedó.

La celda siguiente carga los pesos LoRA: con `TRAIN = False`, el adaptador ya entrenado (`ADAPTER_SRC`); con `TRAIN = True`, el que se acaba de entrenar (`adapter_final` si el entrenamiento terminó; si no, el último checkpoint). Los ficheros de resultados llevan esa etiqueta (`qlora_r16_n2000`, `final`, `step100`, …), así no se mezclan resultados de adaptadores distintos.

In [ ]:
# Cargar los pesos LoRA (el adaptador de la celda 5 está sin entrenar hasta este momento)
if not TRAIN:
    ADAPTER_DIR, TAG = ADAPTER_SRC, ADAPTER_NAME
elif os.path.isfile(f"{OUT_DIR}/adapter_final/adapter_model.safetensors"):
    ADAPTER_DIR, TAG = f"{OUT_DIR}/adapter_final", "final"
elif os.path.isfile(f"{CKPT}/state.pt"):
    ADAPTER_DIR = CKPT
    TAG = f"step{torch.load(f'{CKPT}/state.pt', map_location='cpu', weights_only=False)['step']}"
else:
    raise FileNotFoundError(f"No hay ningún adaptador en {OUT_DIR}: el entrenamiento no ha guardado nada")
set_peft_model_state_dict(model, load_file(f"{ADAPTER_DIR}/adapter_model.safetensors"))
print(f"Adaptador cargado: {ADAPTER_DIR}  (etiqueta: {TAG})")
model.eval()
model.base_model.model.config.use_cache = True

@torch.no_grad()
def generate_answer(row):
    conv = build_conversation(row)
    text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    _, images, _ = process_mm_info(conv, use_audio_in_video=False)
    inputs = to_device(processor(text=text, images=images, return_tensors="pt", padding=True))
    gen = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW, use_cache=True,
                         stop_strings=["Human:", "\nUser:"], tokenizer=processor.tokenizer)
    out = processor.batch_decode(gen[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    return out.strip()

def run_split(rows, path, use_adapter=True, limit=None):
    rows = rows[:limit] if limit else rows
    done = {}
    if os.path.exists(path):
        done = {json.loads(l)["id"]: json.loads(l)["answer"] for l in open(path, encoding="utf-8") if l.strip()}
        print(f"continuando: ya hay {len(done)} respuestas en {path}")
    todo, t0 = [r for r in rows if r["id"] not in done], time.time()
    with open(path, "a", encoding="utf-8") as fh:
        for i, r in enumerate(todo, 1):
            try:
                if use_adapter:
                    a = generate_answer(r)
                else:
                    with model.disable_adapter():
                        a = generate_answer(r)
            except Exception as exc:          # un ejemplo roto no debe parar toda la ejecución
                print(f"  [{r['id'][:12]}] {type(exc).__name__}: {str(exc)[:90]}"); a = ""
            fh.write(json.dumps({"id": r["id"], "answer": a, "lang": TRACK}, ensure_ascii=False) + "\n"); fh.flush()
            if i % 25 == 0 or i == len(todo):
                rate = i / (time.time() - t0)
                print(f"  {i}/{len(todo)} | {rate:.2f}/s | faltan {(len(todo) - i) / rate / 60:.0f} min", flush=True)
    raw = {json.loads(l)["id"]: json.loads(l)["answer"] for l in open(path, encoding="utf-8") if l.strip()}
    return {r["id"]: raw.get(r["id"], "") for r in rows}

eval_rows = dev_rows[:EVAL_LIMIT] if EVAL_LIMIT else dev_rows
if EVAL_LIMIT:                       # en una evaluación parcial, recortar también el fichero gold
    with open(GOLD, "w", encoding="utf-8") as fh:
        for r in eval_rows:
            fh.write(json.dumps({"id": r["id"], "answer": r["answer"]}, ensure_ascii=False) + "\n")

In [ ]:
# a) solo el prompt nuevo, sin fine-tuning (no depende del entrenamiento)
# Valores ya medidos en las primeras 300 preguntas de dev; se usan si se salta v2 con EVAL_LIMIT = 300.
V2_PREV = {"BERTScore-F1": 0.7841, "BERTScore-P": 0.7648, "BERTScore-R": 0.8075, "ROUGE-L": 0.3308}
if EVAL_PROMPT_ONLY:
    raw = run_split(eval_rows, f"{OUT_DIR}/raw_dev_prompt_only.jsonl", use_adapter=False)
    write_preds(f"{OUT_DIR}/prediction_dev_prompt_only.jsonl", {i: clean_answer(a) for i, a in raw.items()})
    scores["v2 (prompt nuevo + limpieza)"] = score(f"{OUT_DIR}/prediction_dev_prompt_only.jsonl", "v2_prompt_only")
elif EVAL_LIMIT == 300:
    scores["v2 (prompt nuevo + limpieza)*"] = V2_PREV
    print("v2 no se repite: se usan los valores medidos antes (* en la tabla). F1 =", V2_PREV["BERTScore-F1"])
else:
    print("EVAL_PROMPT_ONLY = False: se salta v2")

In [ ]:
# b) con el adaptador LoRA
raw = run_split(eval_rows, f"{OUT_DIR}/raw_dev_finetuned_{TAG}.jsonl", use_adapter=True)
write_preds(f"{OUT_DIR}/prediction_dev_finetuned_{TAG}.jsonl", {i: clean_answer(a) for i, a in raw.items()})
scores[f"v3 (fine-tuning, {TAG})"] = score(f"{OUT_DIR}/prediction_dev_finetuned_{TAG}.jsonl", f"v3_finetuned_{TAG}")

In [ ]:
# Comparación (ojo: v0/v1 usan las 1000 preguntas de dev, v2/v3 usan EVAL_LIMIT preguntas;
# * = valor de una ejecución anterior)
print(f"{'Variante':38s} {'F1':>7s} {'P':>7s} {'R':>7s} {'ROUGE-L':>8s}")
for name, s in scores.items():
    print(f"{name:38s} {s['BERTScore-F1']:7.4f} {s['BERTScore-P']:7.4f} {s['BERTScore-R']:7.4f} {s['ROUGE-L']:8.4f}")
json.dump(scores, open(f"{OUT_DIR}/scores_summary.json", "w"), indent=1)

dev_by_id = {r["id"]: r for r in eval_rows}
for p in list(raw.items())[:5]:
    print("\nPregunta:", dev_by_id[p[0]]["question"])
    print("Modelo  :", clean_answer(p[1]))
    print("Gold    :", dev_by_id[p[0]]["answer"])